### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [20]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [23]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 4 PDF files to process

Processing: Aman_Chauhan_Resume.pdf
  ✓ Loaded 1 pages

Processing: Sahil_Neekhra_Apple.pdf
  ✓ Loaded 1 pages

Processing: Srinadh_Data_Engineer.pdf
  ✓ Loaded 1 pages

Processing: Srinidhi_resume.pdf
  ✓ Loaded 2 pages

Total documents loaded: 5


In [25]:
all_pdf_documents


[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-06-05T21:28:11+00:00', 'author': '', 'keywords': '', 'moddate': '2026-06-05T21:28:11+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\Aman_Chauhan_Resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Aman_Chauhan_Resume.pdf', 'file_type': 'pdf'}, page_content='Aman Chauhan\nBackend Engineer — Data Engineer — Generative AI Engineer\n+91 7409535142 chauhanaman909@gmail.com\nProfile\nBackend-focused Software Engineer with 4+ years of experience building scalable data and API-driven systems. Proficient in\nPython, F astAPI, SQL, and distributed data processing, with hands-on experience designinghigh-performance APIs, ETL\npipelines, and cloud-based architectures on Azure.\nExperienced in developingGenerative AI systems, i

In [ ]:
### Another Method to achive that

from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, DirectoryLoader

dir_loader = DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=True
)

documents = dir_loader.load()
documents

100%|██████████| 4/4 [00:00<00:00, 119.54it/s]


[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-06-05T21:28:11+00:00', 'source': '..\\data\\pdf\\Aman_Chauhan_Resume.pdf', 'file_path': '..\\data\\pdf\\Aman_Chauhan_Resume.pdf', 'total_pages': 1, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-06-05T21:28:11+00:00', 'trapped': '', 'modDate': 'D:20260605212811Z', 'creationDate': 'D:20260605212811Z', 'page': 0}, page_content='Aman Chauhan\nBackend Engineer — Data Engineer — Generative AI Engineer\n+91 7409535142 chauhanaman909@gmail.com\nProfile\nBackend-focused Software Engineer with 4+ years of experience building scalable data and API-driven systems. Proficient in\nPython, FastAPI, SQL, and distributed data processing, with hands-on experience designing high-performance APIs, ETL\npipelines, and cloud-based architectures on Azure.\nExperienced in developing Generative AI systems, including a Text-to-SQL platform using LLMs (OpenAI

In [29]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [30]:
chunks=split_documents(all_pdf_documents)
chunks

Split 5 documents into 22 chunks

Example chunk:
Content: Aman Chauhan
Backend Engineer — Data Engineer — Generative AI Engineer
+91 7409535142 chauhanaman909@gmail.com
Profile
Backend-focused Software Engineer with 4+ years of experience building scalable d...
Metadata: {'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-06-05T21:28:11+00:00', 'author': '', 'keywords': '', 'moddate': '2026-06-05T21:28:11+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\Aman_Chauhan_Resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Aman_Chauhan_Resume.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-06-05T21:28:11+00:00', 'author': '', 'keywords': '', 'moddate': '2026-06-05T21:28:11+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\Aman_Chauhan_Resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Aman_Chauhan_Resume.pdf', 'file_type': 'pdf'}, page_content='Aman Chauhan\nBackend Engineer — Data Engineer — Generative AI Engineer\n+91 7409535142 chauhanaman909@gmail.com\nProfile\nBackend-focused Software Engineer with 4+ years of experience building scalable data and API-driven systems. Proficient in\nPython, F astAPI, SQL, and distributed data processing, with hands-on experience designinghigh-performance APIs, ETL\npipelines, and cloud-based architectures on Azure.\nExperienced in developingGenerative AI systems, i

# embedding and vectorStoreDB

In [114]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
import os

In [115]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3475.51it/s]


Model loaded successfully. Embedding dimension: 384


### vectorStore

In [118]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

    def clear_complete_db(self):
        """Drop the named collection and immediately recreate an empty one."""
        self.client.delete_collection(self.collection_name)
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "PDF document embeddings for RAG"},
        )
        print(f"Collection '{self.collection_name}' reset. Documents: {self.collection.count()}")

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [ ]:
# ### To delete old data from persisted file path in chroma
# import gc
# import shutil
# import time
# from pathlib import Path
# from chromadb.api.shared_system_client import SharedSystemClient

# persist_dir = Path("../data/vector_store").resolve()

# # Unlock SQLite: close this client, then stop every Chroma system in this kernel
# if "vectorstore" in globals() and getattr(vectorstore, "client", None) is not None:
#     try:
#         vectorstore.client.close()
#     except Exception:
#         pass
#     vectorstore.client = None
#     vectorstore.collection = None

# for system in list(SharedSystemClient._identifier_to_system.values()):
#     try:
#         system.stop()
#     except Exception:
#         pass
# SharedSystemClient.clear_system_cache()

# gc.collect()
# time.sleep(0.5)

# shutil.rmtree(persist_dir)
# print(f"Deleted {persist_dir}")

Deleted C:\git-clone\RAG---Basic\data\vector_store


In [119]:


### Convert the text to embeddings
texts = [doc.page_content for doc in chunks]
# texts

## Generate embedding

embeddings = embedding_manager.generate_embeddings(texts)

## store in vector database
vector = vectorstore.add_documents(chunks,embeddings)


Generating embeddings for 22 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.67it/s]

Generated embeddings with shape: (22, 384)
Adding 22 documents to vector store...
Successfully added 22 documents to vector store
Total documents in collection: 22


### Retriever Pipeline From VectorStore

In [186]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            # print("results are here - ")
            # print(results)
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    print(f"similarity_score = {similarity_score}, score_threshold = {score_threshold}")
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []



In [187]:
rag_retriever=RAGRetriever(vectorstore,embedding_manager)
rag_retriever.retrieve("how much experience Sahil Neekhra have in data engineering")

Retrieving documents for query: 'how much experience Sahil Neekhra have in data engineering'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 99.00it/s]

Generated embeddings with shape: (1, 384)
similarity_score = 0.1545279622077942, score_threshold = 0.0
similarity_score = -0.11230874061584473, score_threshold = 0.0
similarity_score = -0.142220139503479, score_threshold = 0.0
similarity_score = -0.1924898624420166, score_threshold = 0.0
similarity_score = -0.2598419189453125, score_threshold = 0.0
Retrieved 1 documents (after filtering)


[{'id': 'doc_27791782_4',
  'content': 'SAHIL NEEKHRA \nData Engineer | sahil.neekhra234@gmail.com | +91-7338973598 | LinkedIn | Github \n \nProfessional Summary \nData Engineer with nearly 5 years of experience designing and building scalable batch ETL pipelines and distributed data \nprocessing solutions using Python, PySpark, Scala, Apache Spark, and AWS cloud services. Experienced in developing \nbatch and streaming ETL data pipelines, distributed systems, cloud migration, and event-driven architectures. Delivered \nscalable analytics solutions across Banking and Financial Services (BFSI) and geospatial mapping domains while \nleveraging AI-assisted data engineering workflows to improve developer productivity. \nTechnical Skills \n● Languages & Programming - Python, Scala, SQL, PL/SQL \n● Big Data Technologies – Apache Spark, PySpark, Kafka, Hadoop, Hive, Sqoop, Delta Lake \n● Cloud Technologies – AWS Glue, Amazon EMR, Amazon S3, Amazon Athena, Amazon Redshift, Databricks',
  'meta

### RAG Pipeline- VectorDB To LLM Output Generation

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# print(os.getenv("GROQ_API_KEY"))

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


In [126]:
from langchain_groq import ChatGroq

In [143]:


### Simple RAG pipeline with groq llm

groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(groq_api_key=groq_api_key, model_name="allam-2-7b",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query, retriever ,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "no relevent context found"
    ## generate answer from GROQ LLM

    prompt =f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""

    response = llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [144]:
answer = rag_simple("how much experience Sahil Neekhra have in data engineering", rag_retriever , llm)
answer

Retrieving documents for query: 'how much experience Sahil Neekhra have in data engineering'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 119.69it/s]

Generated embeddings with shape: (1, 384)
results are here - 
{'ids': [['doc_27791782_4', 'doc_38788fb2_13', 'doc_baca5c16_9']], 'embeddings': None, 'documents': [['SAHIL NEEKHRA \nData Engineer | sahil.neekhra234@gmail.com | +91-7338973598 | LinkedIn | Github \n \nProfessional Summary \nData Engineer with nearly 5 years of experience designing and building scalable batch ETL pipelines and distributed data \nprocessing solutions using Python, PySpark, Scala, Apache Spark, and AWS cloud services. Experienced in developing \nbatch and streaming ETL data pipelines, distributed systems, cloud migration, and event-driven architectures. Delivered \nscalable analytics solutions across Banking and Financial Services (BFSI) and geospatial mapping domains while \nleveraging AI-assisted data engineering workflows to improve developer productivity. \nTechnical Skills \n● Languages & Programming - Python, Scala, SQL, PL/SQL \n● Big Data Technologies – Apache Spark, PySpark, Kafka, Hadoop, Hive, Sqo

'Sahil Neekhra has nearly 5 years of experience in data engineering. '

### Enhanced RAG Pipeline Features

In [ ]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output


# Example usage:
query = "how much experience does Sahil Neekhra has as data engineer"
result = rag_advanced(query, rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'how much experience does Sahil Neekhra has as data engineer'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 111.60it/s]

Generated embeddings with shape: (1, 384)
results are here - 
{'ids': [['doc_27791782_4', 'doc_38788fb2_13', 'doc_baca5c16_9']], 'embeddings': None, 'documents': [['SAHIL NEEKHRA \nData Engineer | sahil.neekhra234@gmail.com | +91-7338973598 | LinkedIn | Github \n \nProfessional Summary \nData Engineer with nearly 5 years of experience designing and building scalable batch ETL pipelines and distributed data \nprocessing solutions using Python, PySpark, Scala, Apache Spark, and AWS cloud services. Experienced in developing \nbatch and streaming ETL data pipelines, distributed systems, cloud migration, and event-driven architectures. Delivered \nscalable analytics solutions across Banking and Financial Services (BFSI) and geospatial mapping domains while \nleveraging AI-assisted data engineering workflows to improve developer productivity. \nTechnical Skills \n● Languages & Programming - Python, Scala, SQL, PL/SQL \n● Big Data Technologies – Apache Spark, PySpark, Kafka, Hadoop, Hive, Sqo

Answer: Sahil Neekhra has nearly 5 years of experience as a data engineer. 
Sources: [{'source': 'Sahil_Neekhra_Apple.pdf', 'page': 0, 'score': 0.08966553211212158, 'preview': 'SAHIL NEEKHRA \nData Engineer | sahil.neekhra234@gmail.com | +91-7338973598 | LinkedIn | Github \n \nProfessional Summary \nData Engineer with nearly 5 years of experience designing and building scalable batch ETL pipelines and distributed data \nprocessing solutions using Python, PySpark, Scala, Apache ...'}]
Confidence: 0.08966553211212158
Context Preview: SAHIL NEEKHRA 
Data Engineer | sahil.neekhra234@gmail.com | +91-7338973598 | LinkedIn | Github 
 
Professional Summary 
Data Engineer with nearly 5 years of experience designing and building scalable batch ETL pipelines and distributed data 
processing solutions using Python, PySpark, Scala, Apache 


In [188]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("how much experience does Sahil Neekhra has as data engineer", top_k=3, min_score=0.0, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'how much experience does Sahil Neekhra has as data engineer'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 134.37it/s]

Generated embeddings with shape: (1, 384)
similarity_score = 0.08966553211212158, score_threshold = 0.0
similarity_score = -0.14092981815338135, score_threshold = 0.0
similarity_score = -0.2365320920944214, score_threshold = 0.0
Retrieved 1 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
SAHIL NEEKHRA 
Data Engineer | sahil.neekhra234@gmail.com | +91-7338973598 | LinkedIn | Github 
 
Professional Summary 
Data Engineer with nearly 5 years of experience designing and building scalable batch ETL pipelines and distributed data 
processin

g solutions using Python, PySpark, Scala, Apache Spark, and AWS cloud services. Experienced in developing 
batch and streaming ETL data pipelines, distributed systems, cloud migration, and event-driven architectures. Delivered 
scalable analytics solutions across Banking and Financial Services (BFSI) and geospatial mapping domains while 
leveraging AI-assisted data engineering workflows to improve developer productivity. 
Technical Skills 
● Languages & Programming - Python, Scala, SQL, PL/SQL 
● Big Data Technologies – Apache Spark, PySpark, Kafka, Hadoop, Hive, Sqoop, Delta Lake 
● Cloud Technologies – AWS Glue, Amazon EMR, Amazon S3, Amazon Athena, Amazon Redshift, Databricks

Question: how much experience does Sahil Neekhra has as data engineer

Answer:

Final Answer: Sahil Neekhra has nearly 5 years of experience as a data engineer. 

Citations:
[1] Sahil_Neekhra_Apple.pdf (page 0)
Summary: Sahil Neekhra is an individual with approximately 5 years of expertise in the field of data